1. Train/test split by season
2. Compute ratio features on train and test separately
3. Fill denominator-zero NaNs using train mean only
4. Drop denominator columns
5. Rolling windows
6. Normalization (in modeling notebook)

In [18]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path('data/processed')

outfield = pd.read_parquet(PROCESSED_DIR / 'outfield_clean.parquet')
gk       = pd.read_parquet(PROCESSED_DIR / 'gk_clean.parquet')

ID_COLS = [
    'match_id', 'round', 'match_date', 'home_team', 'away_team',
    'player_id', 'player_name', 'team_id', 'team_name',
    'shirt_number', 'position_id', 'is_goalkeeper', 'season'
]

print(f'Outfield : {outfield.shape[0]:,} rows x {outfield.shape[1]} cols')
print(f'GK       : {gk.shape[0]:,} rows x {gk.shape[1]} cols')
print(f'\nOutfield seasons : {sorted(outfield["season"].unique())}')
print(f'GK seasons       : {sorted(gk["season"].unique())}')
print(f'\nOutfield dtypes:\n{outfield.dtypes.value_counts()}')

Outfield : 21,640 rows x 50 cols
GK       : 2,178 rows x 40 cols

Outfield seasons : ['2021_2022', '2022_2023', '2023_2024', '2024_2025']
GK seasons       : ['2021_2022', '2022_2023', '2023_2024', '2024_2025']

Outfield dtypes:
float64                38
object                  6
int64                   4
datetime64[ns, UTC]     1
bool                    1
Name: count, dtype: int64


In [19]:
# ── Train/test split by season ─────────────────────────────────
TRAIN_SEASONS = ['2021_2022', '2022_2023', '2023_2024']
TEST_SEASON   = ['2024_2025']

outfield_train = outfield[outfield['season'].isin(TRAIN_SEASONS)].copy()
outfield_test  = outfield[outfield['season'].isin(TEST_SEASON)].copy()

gk_train = gk[gk['season'].isin(TRAIN_SEASONS)].copy()
gk_test  = gk[gk['season'].isin(TEST_SEASON)].copy()

print(f'Outfield train : {len(outfield_train):,} rows ({len(outfield_train)/len(outfield)*100:.1f}%)')
print(f'Outfield test  : {len(outfield_test):,} rows ({len(outfield_test)/len(outfield)*100:.1f}%)')
print(f'\nGK train : {len(gk_train):,} rows ({len(gk_train)/len(gk)*100:.1f}%)')
print(f'GK test  : {len(gk_test):,} rows ({len(gk_test)/len(gk)*100:.1f}%)')

Outfield train : 16,198 rows (74.9%)
Outfield test  : 5,442 rows (25.1%)

GK train : 1,630 rows (74.8%)
GK test  : 548 rows (25.2%)


In [20]:
def compute_ratios(df):
    """Compute ratio features for outfield players."""
    df = df.copy()
    df['pass_accuracy']        = df['accurate_passes']    / df['accurate_passes_total']
    df['long_ball_accuracy']   = df['long_balls_accurate'] / df['long_balls_accurate_total']
    df['cross_accuracy']       = df['accurate_crosses']    / df['accurate_crosses_total']
    df['dribble_success_rate'] = df['dribbles_succeeded']  / df['dribbles_succeeded_total']
    df['aerial_win_rate']      = df['aerials_won']         / df['aerials_won_total']
    df['ground_duel_win_rate'] = df['ground_duels_won']    / df['ground_duels_won_total']
    df['shot_accuracy']        = df['ShotsOnTarget']       / (df['ShotsOnTarget'] + df['ShotsOffTarget'])
    return df

def compute_ratios_gk(df):
    """Compute ratio features for goalkeepers."""
    df = df.copy()
    df['pass_accuracy']      = df['accurate_passes']    / df['accurate_passes_total']
    df['long_ball_accuracy'] = df['long_balls_accurate'] / df['long_balls_accurate_total']
    df['aerial_win_rate']      = df['aerials_won']         / df['aerials_won_total']
    df['ground_duel_win_rate'] = df['ground_duels_won']    / df['ground_duels_won_total']
    df['save_rate']          = df['saves']               / (df['saves'] + df['goals_conceded'])
    return df

In [21]:
outfield_train = compute_ratios(outfield_train)
outfield_test  = compute_ratios(outfield_test)
gk_train       = compute_ratios_gk(gk_train)
gk_test        = compute_ratios_gk(gk_test)

# Check NaNs produced
RATIO_COLS_OUTFIELD = [
    'pass_accuracy', 'long_ball_accuracy', 'cross_accuracy',
    'dribble_success_rate', 'aerial_win_rate', 'ground_duel_win_rate', 'shot_accuracy'
]
RATIO_COLS_GK = ['pass_accuracy', 'long_ball_accuracy','aerial_win_rate', 'ground_duel_win_rate', 'save_rate']

print('=== OUTFIELD TRAIN — NaNs from division ===')
for col in RATIO_COLS_OUTFIELD:
    n = outfield_train[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(outfield_train)*100:.1f}%)')

print('\n=== OUTFIELD TEST — NaNs from division ===')
for col in RATIO_COLS_OUTFIELD:
    n = outfield_test[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(outfield_test)*100:.1f}%)')

print('\n=== GK TRAIN — NaNs from division ===')
for col in RATIO_COLS_GK:
    n = gk_train[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(gk_train)*100:.1f}%)')

print('\n=== GK TEST — NaNs from division ===')
for col in RATIO_COLS_GK:
    n = gk_test[col].isna().sum()
    print(f'  {col:<25} {n:>5} NaNs ({n/len(gk_test)*100:.1f}%)')

=== OUTFIELD TRAIN — NaNs from division ===
  pass_accuracy                 0 NaNs (0.0%)
  long_ball_accuracy         3002 NaNs (18.5%)
  cross_accuracy             7655 NaNs (47.3%)
  dribble_success_rate       6836 NaNs (42.2%)
  aerial_win_rate            2949 NaNs (18.2%)
  ground_duel_win_rate        420 NaNs (2.6%)
  shot_accuracy              8426 NaNs (52.0%)

=== OUTFIELD TEST — NaNs from division ===
  pass_accuracy                 0 NaNs (0.0%)
  long_ball_accuracy         1054 NaNs (19.4%)
  cross_accuracy             2544 NaNs (46.7%)
  dribble_success_rate       2310 NaNs (42.4%)
  aerial_win_rate             974 NaNs (17.9%)
  ground_duel_win_rate        140 NaNs (2.6%)
  shot_accuracy              2872 NaNs (52.8%)

=== GK TRAIN — NaNs from division ===
  pass_accuracy                 0 NaNs (0.0%)
  long_ball_accuracy            6 NaNs (0.4%)
  aerial_win_rate            1227 NaNs (75.3%)
  ground_duel_win_rate       1302 NaNs (79.9%)
  save_rate                    46

In [22]:
# ── Compute means from TRAINING data only ─────────────────────
outfield_ratio_means = outfield_train[RATIO_COLS_OUTFIELD].mean()
gk_ratio_means       = gk_train[RATIO_COLS_GK].mean()

print('=== OUTFIELD — training means used for filling ===')
for col, val in outfield_ratio_means.items():
    print(f'  {col:<25} {val:.4f}')

print('\n=== GK — training means used for filling ===')
for col, val in gk_ratio_means.items():
    print(f'  {col:<25} {val:.4f}')

# ── Fill NaNs in BOTH train and test using train means only ───
outfield_train[RATIO_COLS_OUTFIELD] = outfield_train[RATIO_COLS_OUTFIELD].fillna(outfield_ratio_means)
outfield_test[RATIO_COLS_OUTFIELD]  = outfield_test[RATIO_COLS_OUTFIELD].fillna(outfield_ratio_means)
gk_train[RATIO_COLS_GK]             = gk_train[RATIO_COLS_GK].fillna(gk_ratio_means)
gk_test[RATIO_COLS_GK]              = gk_test[RATIO_COLS_GK].fillna(gk_ratio_means)


=== OUTFIELD — training means used for filling ===
  pass_accuracy             0.8164
  long_ball_accuracy        0.4740
  cross_accuracy            0.2307
  dribble_success_rate      0.4811
  aerial_win_rate           0.4808
  ground_duel_win_rate      0.5069
  shot_accuracy             0.4725

=== GK — training means used for filling ===
  pass_accuracy             0.6685
  long_ball_accuracy        0.3584
  aerial_win_rate           0.9826
  ground_duel_win_rate      0.7871
  save_rate                 0.6721


In [23]:
# ── Drop denominator columns — no longer needed ───────────────
DENOM_COLS_OUTFIELD = [
    'accurate_passes_total', 'long_balls_accurate_total', 'accurate_crosses_total',
    'dribbles_succeeded_total', 'aerials_won_total', 'ground_duels_won_total'
]
DENOM_COLS_GK = [
    'accurate_passes_total', 'long_balls_accurate_total',
    'aerials_won_total', 'ground_duels_won_total'
]

outfield_train = outfield_train.drop(columns=DENOM_COLS_OUTFIELD)
outfield_test  = outfield_test.drop(columns=DENOM_COLS_OUTFIELD)
gk_train       = gk_train.drop(columns=DENOM_COLS_GK)
gk_test        = gk_test.drop(columns=DENOM_COLS_GK)

# ── Confirm no NaNs remain in ratio cols ──────────────────────
print('\n=== NaN check after filling ===')
print(f'Outfield train ratio NaNs : {outfield_train[RATIO_COLS_OUTFIELD].isna().sum().sum()}')
print(f'Outfield test ratio NaNs  : {outfield_test[RATIO_COLS_OUTFIELD].isna().sum().sum()}')
print(f'GK train ratio NaNs       : {gk_train[RATIO_COLS_GK].isna().sum().sum()}')
print(f'GK test ratio NaNs        : {gk_test[RATIO_COLS_GK].isna().sum().sum()}')

print(f'\nOutfield train : {outfield_train.shape[0]:,} rows x {outfield_train.shape[1]} cols')
print(f'Outfield test  : {outfield_test.shape[0]:,} rows x {outfield_test.shape[1]} cols')
print(f'GK train       : {gk_train.shape[0]:,} rows x {gk_train.shape[1]} cols')
print(f'GK test        : {gk_test.shape[0]:,} rows x {gk_test.shape[1]} cols')


=== NaN check after filling ===
Outfield train ratio NaNs : 0
Outfield test ratio NaNs  : 0
GK train ratio NaNs       : 0
GK test ratio NaNs        : 0

Outfield train : 16,198 rows x 51 cols
Outfield test  : 5,442 rows x 51 cols
GK train       : 1,630 rows x 41 cols
GK test        : 548 rows x 41 cols
